# 🛠️ Closing the Tool Loop: Error Handling and Evaluation

## Learning Objectives
In this notebook, you will learn:
1. **Closing the Loop** - Send tool results back to the LLM (`tools → agent`) so it can use them, call more tools, or answer
2. **Execution Tracing** - Read the full message trace of a run to see what happened at each step
3. **Error Handling** - Make tool failures reach the LLM instead of crashing the graph: returning errors, and `ToolNode`'s `handle_tool_errors`
4. **Evaluating Tool Use** - Score tool selection, tool input and answer synthesis with an LLM-as-a-judge

## Where this fits
Notebook 03 built an *augmented LLM*: one tool call, then `END`. This notebook changes a single edge so the tool results flow back to the model, and then focuses on what makes that loop robust. The loop as an agent architecture (reason → act → observe, ReAct) is developed in module 03.

## Prerequisites
- `03_augmented_llm_with_tools_langgraph` (state, `ToolNode`, conditional edges)
- `OPENAI_API_KEY` in the project-root `.env`. All tools here are simulated, so no other keys are needed.

| Property | Value |
|---|---|
| Origin | OpenAI function-calling API, June 2023 — conceptual ancestor: Toolformer (Schick et al., 2023) |

---
## 📦 Part 1: Environment Setup

Import the LangGraph and LangChain building blocks, load the API key from `.env`, and initialize the LLM that will power the agent.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and LLM Initialization
# ============================================================================
from typing import Literal

from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, Field
from typing_extensions import Annotated, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from aied_helpers import get_openai_llm

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Initialize the LLM that will decide which tools to call
llm = get_openai_llm(model="gpt-4o-mini")

print(f"🤖 LLM initialized: {llm.model_name}")

🤖 LLM initialized: gpt-4o-mini


---
## 🧰 Part 2: The Toolkit

Three simulated tools, so every run is deterministic and needs no extra API keys. Note that each one already handles its own "not found" case by *returning* a message rather than raising — keep that in mind for Part 5.

### 2.1 🧮 `calculate` — Arithmetic Tool

A simple tool that evaluates a mathematical expression string. Note the `eval()` call is for demonstration only — production code should use a safe math parser instead.

In [2]:
# ============================================================================
# TOOL: calculate - Evaluate a Mathematical Expression
# ============================================================================
@tool
def calculate(expression: str) -> str:
    """Calculate a mathematical expression. Example: calculate('2 + 2')"""
    try:
        result = eval(expression)  # Note: In production, use a safe math parser
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error calculating: {e}"

### 2.2 🌤️ `get_weather` — Weather Lookup Tool

A tool that looks up weather for a small set of hardcoded cities. Cities outside the lookup table return a friendly "not available" message instead of raising an error.

In [3]:
# ============================================================================
# TOOL: get_weather - Simulated Weather Lookup
# ============================================================================
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # Simulated weather data
    weather_data = {
        "new york": "72°F, Sunny",
        "london": "58°F, Cloudy",
        "tokyo": "68°F, Clear",
        "paris": "65°F, Partly Cloudy",
    }
    city_lower = city.lower()
    if city_lower in weather_data:
        return f"Weather in {city}: {weather_data[city_lower]}"
    return f"Weather data not available for {city}"

### 2.3 🔍 `search_web` — Simulated Web Search Tool

A tool that simulates a web search by matching the query against a small hardcoded results table.

In [4]:
# ============================================================================
# TOOL: search_web - Simulated Web Search
# ============================================================================
@tool
def search_web(query: str) -> str:
    """Simulate a web search for a query."""
    # Simulated search results
    search_results = {
        "python programming": "Python is a high-level programming language known for its readability and versatility.",
        "latest news": "Today's top news: AI continues to advance, impacting various industries worldwide.",
        "best restaurants in new york": "Top restaurants in New York include Le Bernardin, Per Se, and Eleven Madison Park.",
    }
    query_lower = query.lower()
    if query_lower in search_results:
        return f"Search results for '{query}': {search_results[query_lower]}"
    return f"No search results found for '{query}'"

---
## 🧩 Part 3: Closing the Loop

### Key Concepts:
- **`AgentState`**: A `TypedDict` holding the running list of messages; the `add_messages` reducer appends instead of overwriting, which is what lets the conversation accumulate across agent ↔ tool round trips
- **`should_continue`**: Inspects the last message and routes to `tools` if it contains tool calls, otherwise ends the run
- **The one new edge**: `tools → agent`. In notebook 03 it was `tools → END`

In [5]:
# ============================================================================
# STATE: AgentState - Message History Schema
# ============================================================================
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

### `create_tool_agent` — Building the Tool-Calling Loop

This function assembles the graph for any list of tools:
1. Bind the tools to the LLM with `bind_tools()` so it can emit tool calls
2. An `agent` node that invokes the LLM
3. A `tools` node (`ToolNode`) that executes whichever tools the LLM requested — `handle_tool_errors` is explained in Part 5
4. Conditional edges that loop back to `agent` after tool execution, until the LLM answers without any tool calls

In [6]:
# ============================================================================
# GRAPH: create_tool_agent - Assemble the Tool-Calling Loop
# ============================================================================
def create_tool_agent(tools, handle_tool_errors=True):
    """Create a tool-calling agent over the given tools."""
    llm_with_tools = llm.bind_tools(tools)

    def agent_node(state: AgentState) -> dict:
        # Generate a response using the LLM with tool access
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    def should_continue(state: AgentState) -> Literal["tools", "end"]:
        """Route to the tools if the LLM asked for any, otherwise finish."""
        last_message = state["messages"][-1]
        if not getattr(last_message, "tool_calls", None):
            return "end"
        return "tools"

    graph = StateGraph(AgentState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", ToolNode(tools, handle_tool_errors=handle_tool_errors))
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
    graph.add_edge("tools", "agent")  # the edge that closes the loop

    return graph.compile()


agent = create_tool_agent([calculate, get_weather, search_web])
print("✅ Agent compiled")

✅ Agent compiled


---
## ▶️ Part 4: Running the Agent and Reading the Trace

### 4.1 💬 Sample Queries

Each query prints the final answer and how many messages accumulated in state (user message + tool-call requests + tool results + final answer). A two-part question produces more messages because the model calls two tools.

In [7]:
# ============================================================================
# RUN: A Few Sample Queries
# ============================================================================
queries = [
    "What's 25 * 17?",
    "What's the weather in Tokyo?",
    "What's 100 / 4 and what's the weather in London?",
]

for query in queries:
    result = agent.invoke({"messages": [HumanMessage(content=query)]})
    print(f"Query: {query}")
    print(f"Response: {result['messages'][-1].content}")
    print(f"Total messages: {len(result['messages'])}")
    print("-" * 40)

Query: What's 25 * 17?
Response: The result of \( 25 \times 17 \) is 425.
Total messages: 4
----------------------------------------
Query: What's the weather in Tokyo?
Response: The weather in Tokyo is currently 68°F and clear.
Total messages: 4
----------------------------------------
Query: What's 100 / 4 and what's the weather in London?
Response: The result of \( 100 / 4 \) is 25.0. 

As for the weather in London, it is currently 58°F and cloudy.
Total messages: 5
----------------------------------------


### 4.2 🔎 Inspecting the Full Message Trace

Rather than only the final answer, walk every message in the resulting state — the human query, the AI's tool-call requests, and each tool's result — to see exactly how the loop unfolded. This trace is also the input to the evaluation in Part 6.

In [8]:
# ============================================================================
# TRACE: Walk Every Message of One Run
# ============================================================================
trace_result = agent.invoke(
    {"messages": [HumanMessage(content="Calculate 15% of 250 and check weather in Paris")]}
)

for i, msg in enumerate(trace_result["messages"]):
    print(f"\n[{i}] {type(msg).__name__}:")
    if isinstance(msg, HumanMessage):
        print(f"  Content: {msg.content}")
    elif isinstance(msg, AIMessage):
        if msg.tool_calls:
            print(f"  Tool calls: {len(msg.tool_calls)}")
            for tc in msg.tool_calls:
                print(f"    - {tc['name']}({tc['args']})")
        else:
            print(f"  Content: {msg.content}")
    elif isinstance(msg, ToolMessage):
        print(f"  Tool: {msg.name}")
        print(f"  Result: {msg.content}")


[0] HumanMessage:
  Content: Calculate 15% of 250 and check weather in Paris

[1] AIMessage:
  Tool calls: 2
    - calculate({'expression': '0.15 * 250'})
    - get_weather({'city': 'Paris'})

[2] ToolMessage:
  Tool: calculate
  Result: The result of 0.15 * 250 is 37.5

[3] ToolMessage:
  Tool: get_weather
  Result: Weather in Paris: 65°F, Partly Cloudy

[4] AIMessage:
  Content: 15% of 250 is 37.5. 

The current weather in Paris is 65°F and partly cloudy.


---
## ⚠️ Part 5: Error Handling in Tools

Tools fail: division by zero, an unknown ID, a timed-out API. What matters is *where the failure goes*. If it reaches the LLM as a `ToolMessage`, the model can explain the problem or retry with better arguments. If it escapes as an exception, the whole graph run crashes.

There are two ways to get it to the LLM:
1. **The tool returns the error** — it catches the failure itself and returns a descriptive string (5.1)
2. **`ToolNode` catches it** — the tool raises, and `handle_tool_errors` turns the exception into an error `ToolMessage` (5.2)

### 5.1 ➕ A Tool That Returns Its Error

In [9]:
# ============================================================================
# TOOL: divide - Division With Guarded Zero-Division Handling
# ============================================================================
@tool
def divide(a: float, b: float) -> str:
    """Divide two numbers."""
    if b == 0:
        return "Error: Division by zero"
    result = a / b
    return f"The result of {a} divided by {b} is {result}"

In [10]:
# ============================================================================
# RUN: The error string flows back to the LLM like any other result
# ============================================================================
divide_agent = create_tool_agent([divide])

for query in ["Divide 100 by 5", "Divide 100 by 0"]:
    result = divide_agent.invoke({"messages": [HumanMessage(content=query)]})
    print(f"Query: {query}")
    print(f"Response: {result['messages'][-1].content}")
    print("-" * 40)

Query: Divide 100 by 5
Response: The result of dividing 100 by 5 is 20.
----------------------------------------
Query: Divide 100 by 0
Response: Division by zero is undefined in mathematics, so it's not possible to divide 100 by 0. If you have any other calculations or questions, feel free to ask!
----------------------------------------


### 5.2 🧯 A Tool That Raises — and What `ToolNode` Does With It

Real tools often *raise* (a client library throws, a lookup fails). `ToolNode` has a `handle_tool_errors` setting, and its default is narrower than it looks:

| `handle_tool_errors` | Bad arguments from the model | Exception raised inside the tool |
|---|---|---|
| default | returned to the LLM as an error `ToolMessage` | **re-raised: the graph run crashes** |
| `True` | returned as an error `ToolMessage` | returned as an error `ToolMessage` |
| `False` | re-raised | re-raised |

The cell below shows this without calling the LLM: a one-node graph runs the `ToolNode` on a hand-written tool call, so the result is deterministic.

In [11]:
# ============================================================================
# DEMO: The same failing call under each handle_tool_errors setting
# ============================================================================
@tool
def get_stock_price(ticker: str) -> str:
    """Get the latest price for a stock ticker symbol, e.g. 'AAPL'."""
    prices = {"AAPL": 227.50, "MSFT": 431.20}
    if ticker.upper() not in prices:
        raise ValueError(f"Unknown ticker {ticker!r}; supported: {sorted(prices)}")
    return f"{ticker.upper()}: ${prices[ticker.upper()]}"


def run_tool_node(tool_node, args):
    """Run a ToolNode on one hand-written tool call; report what came out."""
    g = StateGraph(AgentState)
    g.add_node("tools", tool_node)
    g.add_edge(START, "tools")
    g.add_edge("tools", END)
    call = AIMessage(content="", tool_calls=[{"name": "get_stock_price", "args": args, "id": "call_1"}])
    try:
        msg = g.compile().invoke({"messages": [call]})["messages"][-1]
        return f"ToolMessage(status={msg.status!r}): {msg.content}"
    except Exception as e:
        return f"💥 RAISED {type(e).__name__}: {e}"


settings = {
    "default": ToolNode([get_stock_price]),
    "True": ToolNode([get_stock_price], handle_tool_errors=True),
    "False": ToolNode([get_stock_price], handle_tool_errors=False),
}
for label, node in settings.items():
    print(f"handle_tool_errors={label}")
    print(f"  right arguments   → {run_tool_node(node, {'ticker': 'AAPL'})}")
    print(f"  bad arguments   → {run_tool_node(node, {'symbol': 'AAPL'})}")
    print(f"  tool raises     → {run_tool_node(node, {'ticker': 'NVDA'})}")
    print()

handle_tool_errors=default
  right arguments   → ToolMessage(status='success'): AAPL: $227.5
  bad arguments   → ToolMessage(status='error'): Error invoking tool 'get_stock_price' with kwargs {'symbol': 'AAPL'} with error:
 ticker: Field required
 Please fix the error and try again.
  tool raises     → 💥 RAISED ValueError: Unknown ticker 'NVDA'; supported: ['AAPL', 'MSFT']

handle_tool_errors=True
  right arguments   → ToolMessage(status='success'): AAPL: $227.5
  bad arguments   → ToolMessage(status='error'): Error: ToolInvocationError("Error invoking tool 'get_stock_price' with kwargs {'symbol': 'AAPL'} with error:\n ticker: Field required\n Please fix the error and try again.")
 Please fix your mistakes.
  tool raises     → ToolMessage(status='error'): Error: ValueError("Unknown ticker 'NVDA'; supported: ['AAPL', 'MSFT']")
 Please fix your mistakes.

handle_tool_errors=False
  right arguments   → ToolMessage(status='success'): AAPL: $227.5
  bad arguments   → 💥 RAISED ToolInvocation

With `handle_tool_errors=True` — which `create_tool_agent` uses by default — the model sees the exception text and can recover, for example by telling the user which tickers are supported:

In [12]:
# ============================================================================
# RUN: The agent recovers from a raised exception
# ============================================================================
stock_agent = create_tool_agent([get_stock_price], handle_tool_errors=True)

result = stock_agent.invoke({"messages": [HumanMessage(content="What's the price of NVDA?")]})
for msg in result["messages"]:
    if isinstance(msg, ToolMessage):
        print(f"🔧 ToolMessage (status={msg.status}): {msg.content[:100]}")
print(f"\n🤖 Response: {result['messages'][-1].content}")

🔧 ToolMessage (status=error): Error: ValueError("Unknown ticker 'NVDA'; supported: ['AAPL', 'MSFT']")
 Please fix your mistakes.
🔧 ToolMessage (status=success): AAPL: $227.5
🔧 ToolMessage (status=success): MSFT: $431.2

🤖 Response: I was unable to retrieve the price for NVDA (NVIDIA). However, here are the latest prices for the available stocks:

- AAPL (Apple): $227.5
- MSFT (Microsoft): $431.2

If you need information on other stocks or have any other questions, feel free to ask!


---
## 📏 Part 6: Evaluating Tool Use with an LLM-as-a-Judge

A run that finishes is not necessarily a good run. For a tool-using agent, evaluate three things separately:
1. **Tool selection** — did it pick the right tool(s)?
2. **Tool input** — were the arguments well-formed and relevant?
3. **Synthesis** — did the final answer correctly use what the tools returned?

The trace from 4.2 already contains all three. Below, a judge LLM reads that trace and returns structured scores via `with_structured_output`.

> **Caveat**: the judge here is the same model as the agent, which tends to be lenient on its own work. In practice, use a different (ideally stronger) model as the judge and check its scores against a few human-graded runs.

In [13]:
# ============================================================================
# EVALUATION: Structured scores from a judge LLM
# ============================================================================
class ToolUseEvaluation(BaseModel):
    """Schema for evaluating the agent's tool use and final answer."""
    tool_selection_score: int = Field(description="Score 1-5 on whether the agent chose the correct tools for the task.")
    tool_input_score: int = Field(description="Score 1-5 on how well-formed and relevant the tool inputs were.")
    synthesis_quality_score: int = Field(description="Score 1-5 on how well the final answer integrates the tool outputs.")
    justification: str = Field(description="A brief justification for the scores.")


judge_llm = llm.with_structured_output(ToolUseEvaluation)

# Flatten the trace from 4.2 into text the judge can read
conversation_trace = "\n".join(
    f"{m.type}: {m.content or ''} {getattr(m, 'tool_calls', '') or ''}"
    for m in trace_result["messages"]
)

evaluation = judge_llm.invoke(
    f"""You are an expert judge of AI agents. Evaluate the agent's tool use in the
conversation trace below on each criterion (1-5), with a brief justification.

Conversation Trace:
```
{conversation_trace}
```"""
)

print(conversation_trace)
print("\n📏 Evaluation:")
for field, value in evaluation.model_dump().items():
    print(f"  {field}: {value}")

human: Calculate 15% of 250 and check weather in Paris 
ai:  [{'name': 'calculate', 'args': {'expression': '0.15 * 250'}, 'id': 'call_pk9nM84Jd7ulpZAwCmmCqkQr', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': 'Paris'}, 'id': 'call_yAMMmxtr51FCyDbnT13kRHhB', 'type': 'tool_call'}]
tool: The result of 0.15 * 250 is 37.5 
tool: Weather in Paris: 65°F, Partly Cloudy 
ai: 15% of 250 is 37.5. 

The current weather in Paris is 65°F and partly cloudy. 

📏 Evaluation:
  tool_selection_score: 5
  tool_input_score: 5
  synthesis_quality_score: 5
  justification: The agent effectively selected the appropriate tools for both calculations and weather retrieval, demonstrating a clear understanding of the tasks. The inputs for both tools were well-formed and relevant, directly addressing the user's requests. The final answer synthesized the outputs from both tools seamlessly, providing a clear and concise response.


---
## 📝 Summary

In this notebook, we learned:

### 1. Closing the Loop
- One edge — `tools → agent` instead of `tools → END` — lets the LLM read tool results and decide what to do next
- `create_tool_agent()` wires an `agent` node, a `ToolNode` and a `should_continue` router into that loop

### 2. Execution Tracing
- The message list is the trace: `HumanMessage` → `AIMessage` with `tool_calls` → `ToolMessage` results → final `AIMessage`

### 3. Error Handling
- A tool can **return** its error as a string, or **raise** and let `ToolNode` catch it
- `ToolNode`'s default only catches bad arguments; set `handle_tool_errors=True` so exceptions raised inside tools reach the LLM instead of crashing the run

### 4. Evaluating Tool Use
- Score tool selection, tool input and synthesis separately, from the trace
- `with_structured_output` gives machine-readable scores; use an independent judge model in practice

### Next Steps
- **Module 03 — ReAct agents**: the tool loop as an agent architecture (reason → act → observe)
- **`05_hosted_vs_client_side_tools`**: what changes when the provider runs the tool instead of your code